# YieldSAT: EDA

YieldSAT is the yield-regression benchmark for this repository. It is a CVPR 2026 multimodal crop-yield dataset with combine-harvester yield maps, Sentinel-2 time series, weather, soil, and topography across Argentina, Brazil, Germany, and Uruguay.

This notebook focuses on the format this repo uses: the ML-ready `preprocessed-24-ts` NetCDF release. The loader aggregates YieldSAT's pixel-level rows to one field-level sample per `field_shared_name`, because the frozen encoders in this repository produce one embedding per sample.

References:

- Project page: https://yieldsat.github.io/
- CVF paper page: https://openaccess.thecvf.com/content/CVPR2026/html/Miranda_YieldSAT_A_Multimodal_Benchmark_Dataset_for_High-Resolution_Crop_Yield_Prediction_CVPR_2026_paper.html
- arXiv: https://arxiv.org/abs/2604.00940
- Official overview notebook: https://yieldsat.github.io/notebooks/yieldsat-overview-notebook.html


## Setup

In [1]:
from pathlib import Path
import sys
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import xarray as xr
except ImportError:
    xr = None

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 120

REPO = Path.cwd().resolve()
while not (REPO / "src").exists() and REPO != REPO.parent:
    REPO = REPO.parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

YIELDSAT = REPO / "data" / "input" / "yieldsat"
PREPROCESSED = YIELDSAT / "preprocessed-24-ts"
SOURCE_COPY = REPO / "temp" / "YieldSAT"
COUNTRIES = ["Argentina", "Brazil", "Germany", "Uruguay"]
NC_NAME = "merge_s2-soil-dem-weather-coords.nc"

print("repo:", REPO)
print("staged YieldSAT:", YIELDSAT)
print("source copy:", SOURCE_COPY if SOURCE_COPY.exists() else "not found")

repo: /Users/akshithchowdary/Developer/Projects/org/abe/robustness
staged YieldSAT: /Users/akshithchowdary/Developer/Projects/org/abe/robustness/data/input/yieldsat
source copy: /Users/akshithchowdary/Developer/Projects/org/abe/robustness/temp/YieldSAT


## 1. What is staged?

In [2]:
rows = []
for country in COUNTRIES:
    nc = PREPROCESSED / country / NC_NAME
    src_zip = SOURCE_COPY / "Preprocessed" / country / f"{country}.zip"
    raw_zip = SOURCE_COPY / "Raw" / country / f"{country}.zip"
    rows.append(
        {
            "country": country,
            "netcdf_staged": nc.exists(),
            "netcdf_size_gib": nc.stat().st_size / 1024**3 if nc.exists() else np.nan,
            "preprocessed_zip_present": src_zip.exists(),
            "preprocessed_zip_size_gib": src_zip.stat().st_size / 1024**3 if src_zip.exists() else np.nan,
            "raw_zip_present": raw_zip.exists(),
            "raw_zip_size_gib": raw_zip.stat().st_size / 1024**3 if raw_zip.exists() else np.nan,
        }
    )
staging = pd.DataFrame(rows)
staging

,country,netcdf_staged,netcdf_size_gib,preprocessed_zip_present,preprocessed_zip_size_gib,raw_zip_present,raw_zip_size_gib
0,Argentina,False,NaN,True,2.096587,True,7.355427
1,Brazil,False,NaN,True,1.706986,True,3.347129
2,Germany,False,NaN,True,0.270813,True,1.929876
3,Uruguay,False,NaN,True,0.843817,True,5.494830


In [3]:
if SOURCE_COPY.exists():
    zip_rows = []
    for z in sorted((SOURCE_COPY / "Preprocessed").glob("*/*.zip")):
        with zipfile.ZipFile(z) as archive:
            members = archive.infolist()
            zip_rows.append(
                {
                    "zip": str(z.relative_to(REPO)),
                    "members": len(members),
                    "uncompressed_gib": sum(m.file_size for m in members) / 1024**3,
                    "first_member": members[0].filename if members else None,
                }
            )
    display(pd.DataFrame(zip_rows))
else:
    print("No temp/YieldSAT source copy is present.")

,zip,members,uncompressed_gib,first_member
0,temp/YieldSAT/Preprocessed/Argentina/Argentina...,1,58.543949,merge_s2-soil-dem-weather-coords.nc
1,temp/YieldSAT/Preprocessed/Brazil/Brazil.zip,1,46.826937,merge_s2-soil-dem-weather-coords.nc
2,temp/YieldSAT/Preprocessed/Germany/Germany.zip,1,6.699405,merge_s2-soil-dem-weather-coords.nc
3,temp/YieldSAT/Preprocessed/Uruguay/Uruguay.zip,1,23.928998,merge_s2-soil-dem-weather-coords.nc


The repository intentionally stages only the ML-ready NetCDF release. The raw flexible release remains outside `data/input/yieldsat/`; see the README provenance section for the filtering rationale.

## 2. Open one country file

In [4]:
available = [PREPROCESSED / c / NC_NAME for c in COUNTRIES if (PREPROCESSED / c / NC_NAME).exists()]
if not available:
    print("No NetCDFs are staged yet. Run the README extraction command first.")
    ds = None
else:
    if xr is None:
        raise ImportError("xarray is required to inspect YieldSAT NetCDF files")
    ds = xr.open_dataset(available[0])
    print("opened:", available[0].relative_to(REPO))
    display(ds)

No NetCDFs are staged yet. Run the README extraction command first.


In [5]:
if ds is not None:
    vars_df = pd.DataFrame(
        [
            {"name": name, "dims": ",".join(var.dims), "shape": tuple(var.shape), "dtype": str(var.dtype)}
            for name, var in ds.data_vars.items()
        ]
    )
    display(vars_df)
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


## 3. Decode categorical metadata

In [6]:
def decode_attrs(attrs):
    out = {}
    for key, value in attrs.items():
        try:
            out[int(key)] = value
        except (TypeError, ValueError):
            pass
    return out

if ds is not None:
    for var in ["country", "crop", "year", "farm_identifier", "field_shared_name"]:
        if var in ds:
            mapping = decode_attrs(ds[var].attrs)
            print(var, "mapping entries:", len(mapping))
            if mapping:
                display(pd.DataFrame(list(mapping.items())[:10], columns=["code", "value"]))
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


## 4. Pixel-level target distribution

In [7]:
if ds is not None:
    target = ds["target"].values.astype(float)
    print(pd.Series(target).describe())
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(target[np.isfinite(target)], bins=60, color="#4C78A8", alpha=0.85)
    ax.set_title("Pixel-level yield target distribution")
    ax.set_xlabel("yield (t/ha)")
    ax.set_ylabel("pixels")
    plt.show()
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


## 5. Field-level aggregation used by this repo

In [8]:
def decoded_values(ds, var):
    values = ds[var].values
    mapping = decode_attrs(ds[var].attrs)
    if mapping:
        return np.asarray([mapping.get(int(v), str(int(v))) for v in values], dtype=object)
    return values.astype(str)

if ds is not None:
    fields = decoded_values(ds, "field_shared_name")
    crops = decoded_values(ds, "crop") if "crop" in ds else np.full(len(fields), "unknown", dtype=object)
    years = decoded_values(ds, "year") if "year" in ds else np.full(len(fields), "unknown", dtype=object)
    pix = pd.DataFrame({"field": fields, "crop": crops, "year": years, "target": ds["target"].values.astype(float)})
    field_df = pix.groupby("field").agg(
        pixels=("target", "size"),
        yield_mean=("target", "mean"),
        yield_std=("target", "std"),
        crop=("crop", "first"),
        year=("year", "first"),
    ).reset_index()
    display(field_df.head())
    print(field_df[["pixels", "yield_mean", "yield_std"]].describe())
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


In [9]:
if ds is not None:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].hist(field_df["yield_mean"], bins=40, color="#59A14F", alpha=0.85)
    ax[0].set_title("Field-mean yield")
    ax[0].set_xlabel("yield (t/ha)")
    ax[0].set_ylabel("fields")
    ax[1].hist(field_df["pixels"], bins=40, color="#F28E2B", alpha=0.85)
    ax[1].set_title("Pixels per field")
    ax[1].set_xlabel("10 m pixels")
    ax[1].set_ylabel("fields")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


## 6. Band layout for the frozen encoders

In [10]:
if ds is not None:
    bands = [str(b) for b in ds["band"].values]
    s2 = ["B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B11", "B12"]
    aux = ["temp_mean", "total_prec", "dem"]
    layout = pd.DataFrame(
        {
            "band": bands,
            "used_as_s2": [b in s2 for b in bands],
            "used_as_climate": [b in aux for b in bands],
        }
    )
    display(layout[layout.used_as_s2 | layout.used_as_climate])
else:
    print("Skipped: no staged NetCDF.")

Skipped: no staged NetCDF.


## 7. Check the repository loader

In [11]:
from dataio.get_input import load_yieldsat

if all((PREPROCESSED / c / NC_NAME).exists() for c in COUNTRIES):
    bench = load_yieldsat(root=YIELDSAT.parent, max_samples=20, shuffle=True, seed=0)
    print("samples:", bench.n_samples)
    print("task:", bench.task)
    print("s2:", bench.s2.shape, bench.s2_bands)
    print("s1:", bench.s1.shape, "mask sum", bench.s1_mask.sum())
    print("climate:", bench.climate.shape, bench.climate_bands)
    print("labels:", bench.labels[:5], "range", (float(np.min(bench.labels)), float(np.max(bench.labels))))
    print("groups:", sorted(np.unique(bench.groups).tolist()))
else:
    print("Skipped loader smoke test: all four country NetCDFs are not staged yet.")

Skipped loader smoke test: all four country NetCDFs are not staged yet.


## 8. Benchmark interpretation

`yield-reg` is a regression task. The strict holdouts are countries: Argentina, Brazil, Germany, and Uruguay. For each held-out country, probes train on the other countries and evaluate transfer to the held-out country. Target-budget rows then add a small number of held-out-country labels to the source training set.

Important caveats:

- YieldSAT is originally pixel-level; this repo's task is field-level because each frozen encoder call produces one embedding per sample.
- Sentinel-1 is absent, so `sensor_off_s1` is a sanity condition here.
- The loader uses Sentinel-2 plus `temp_mean`, `total_prec`, and `dem`; the rest of the 120 NetCDF bands remain available in the staged data but are not consumed by the current encoders.
